# Monitoring & Consumption Layer

### Project setup

In [4]:
project = !gcloud config get-value project
PROJECT_ID = project[0]
BUCKET = PROJECT_ID
REGION = "us-central1"

GCS_DATA_PATH = f"gs://{BUCKET}/data/input/synthetic_fashion_demand_features.csv"
NOTIFICATION_EMAILS = ["sandeep.raju.anantha@gmail.com"]  # <- match whatever you set in notebook 6
MODEL_DISPLAY_NAME = "fashion_demand_forecasting"
ENDPOINT_NAME = "fashion_demand_endpoint"
BQ_DATASET = "forecasting"
PUBSUB_TOPIC_ID = "pipeline-notifications"

from google.cloud import aiplatform, bigquery

aiplatform.init(project=PROJECT_ID, location=REGION)
bq_client = bigquery.Client(project=PROJECT_ID)

### Consumption layer: reading the forecasts

- The write runs on every pipeline execution, regardless of whether that run's candidate was promoted
- data at `{PROJECT_ID}.{BQ_DATASET}.latest_forecasts` via BigQuery's native connector 

In [5]:
import pandas as pd

latest_rows = list(bq_client.query(f"""
    SELECT *
    FROM `{PROJECT_ID}.{BQ_DATASET}.latest_forecasts`
    ORDER BY date, sku_id
    LIMIT 50
""").result())
latest_df = pd.DataFrame.from_records([dict(row.items()) for row in latest_rows])

print(f"{len(latest_df)} rows in latest_forecasts")
latest_df.head(10)

50 rows in latest_forecasts


,sku_id,date,predicted_sales_qty,model_display_name,model_resource_name,generated_at
0,SKU00000,2024-12-30,14.304440,fashion_demand_forecasting,projects/456832640267/locations/us-central1/mo...,2026-09-11 03:07:30.418443
1,SKU00001,2024-12-30,27.841751,fashion_demand_forecasting,projects/456832640267/locations/us-central1/mo...,2026-09-11 03:07:30.418443
2,SKU00002,2024-12-30,9.424680,fashion_demand_forecasting,projects/456832640267/locations/us-central1/mo...,2026-09-11 03:07:30.418443
3,SKU00003,2024-12-30,10.660276,fashion_demand_forecasting,projects/456832640267/locations/us-central1/mo...,2026-09-11 03:07:30.418443
4,SKU00004,2024-12-30,10.408463,fashion_demand_forecasting,projects/456832640267/locations/us-central1/mo...,2026-09-11 03:07:30.418443
5,SKU00005,2024-12-30,22.687231,fashion_demand_forecasting,projects/456832640267/locations/us-central1/mo...,2026-09-11 03:07:30.418443
6,SKU00006,2024-12-30,9.564734,fashion_demand_forecasting,projects/456832640267/locations/us-central1/mo...,2026-09-11 03:07:30.418443
7,SKU00007,2024-12-30,37.104698,fashion_demand_forecasting,projects/456832640267/locations/us-central1/mo...,2026-09-11 03:07:30.418443
8,SKU00008,2024-12-30,78.832710,fashion_demand_forecasting,projects/456832640267/locations/us-central1/mo...,2026-09-11 03:07:30.418443
9,SKU00009,2024-12-30,2.543262,fashion_demand_forecasting,projects/456832640267/locations/us-central1/mo...,2026-09-11 03:07:30.418443


### Sanity-checking the history table

In [6]:
history_rows = list(bq_client.query(f"""
    SELECT
      generated_at,
      model_display_name,
      COUNT(*) AS n_rows,
      COUNT(DISTINCT sku_id) AS n_skus
    FROM `{PROJECT_ID}.{BQ_DATASET}.forecast_history`
    GROUP BY generated_at, model_display_name
    ORDER BY generated_at DESC
    LIMIT 20
""").result())
history_summary_df = pd.DataFrame.from_records([dict(row.items()) for row in history_rows])

history_summary_df

,generated_at,model_display_name,n_rows,n_skus
0,2026-09-11 03:07:30.418443+00:00,fashion_demand_forecasting,2400,300
1,2026-09-11 01:59:53.662472+00:00,fashion_demand_forecasting,300,300
2,2026-09-09 17:46:53.168160+00:00,fashion_demand_forecasting,2,2


# Model monitoring: skew & drift on the deployed endpoint

In [10]:
CREATE_MONITORING_JOB = True  # <- flip to True once you want this running continuously

from google.cloud.aiplatform import model_monitoring as mm

endpoints = aiplatform.Endpoint.list(filter=f'display_name="{ENDPOINT_NAME}"')
if not endpoints:
    print(f"No endpoint named {ENDPOINT_NAME} yet -- deploy a model first (notebook 6), then come back here.")
elif CREATE_MONITORING_JOB:
    endpoint = endpoints[0]
    deployed_model_ids = [m.id for m in endpoint.list_models()]

    MONITORED_NUMERIC_FEATURES = ["price", "discount_pct", "competitor_price_index"]
    MONITORED_CATEGORICAL_FEATURES = ["category", "promo_flag"]
    monitored_features = MONITORED_NUMERIC_FEATURES + MONITORED_CATEGORICAL_FEATURES

    objective_config = mm.ObjectiveConfig(
        skew_detection_config=mm.SkewDetectionConfig(
            data_source=GCS_DATA_PATH,
            target_field="sales_qty",
            skew_thresholds=0.3,  # one threshold applied across all monitored features
            data_format="csv",
        ),
        drift_detection_config=mm.DriftDetectionConfig(
            drift_thresholds={f: 0.3 for f in monitored_features},
        ),
    )

    monitoring_job = aiplatform.ModelDeploymentMonitoringJob.create(
        display_name=f"{ENDPOINT_NAME}-skew-drift-monitoring",
        endpoint=endpoint,
        deployed_model_ids=deployed_model_ids,
        logging_sampling_strategy=mm.RandomSampleConfig(sample_rate=0.8),
        schedule_config=mm.ScheduleConfig(monitor_interval=24),  # hours between analysis runs
        alert_config=mm.EmailAlertConfig(
            user_emails=NOTIFICATION_EMAILS,
            enable_logging=True,
        ),
        objective_configs=objective_config,
    )
    print(f"Created monitoring job: {monitoring_job.resource_name}")
else:
    print("CREATE_MONITORING_JOB is False -- endpoint found, nothing created, nothing billed.")

Creating ModelDeploymentMonitoringJob
ModelDeploymentMonitoringJob created. Resource name: projects/456832640267/locations/us-central1/modelDeploymentMonitoringJobs/656572964799512576
To use this ModelDeploymentMonitoringJob in another session:
mdm_job = aiplatform.ModelDeploymentMonitoringJob('projects/456832640267/locations/us-central1/modelDeploymentMonitoringJobs/656572964799512576')
View Model Deployment Monitoring Job:
https://console.cloud.google.com/agent-platform/locations/us-central1/model-deployment-monitoring/656572964799512576?project=456832640267
Created monitoring job: projects/456832640267/locations/us-central1/modelDeploymentMonitoringJobs/656572964799512576


### Managing an existing monitoring job

Pausing costs nothing further; deleting removes the resource entirely.

In [11]:
monitoring_jobs = aiplatform.ModelDeploymentMonitoringJob.list(
    filter=f'display_name="{ENDPOINT_NAME}-skew-drift-monitoring"'
)
for job in monitoring_jobs:
    print(job.resource_name, job.state)

job.pause()    # stop future analysis runs, keep the resource and its history
# job.resume()   # resume a paused job
# job.delete()   # remove it entirely

projects/456832640267/locations/us-central1/modelDeploymentMonitoringJobs/656572964799512576 2


resource name: projects/456832640267/locations/us-central1/modelDeploymentMonitoringJobs/656572964799512576

# Realized vs. predicted accuracy, once actuals land

In [7]:
CREATE_ACCURACY_SCHEDULE = False  # <- flip to True once an `actuals` table is actually being populated

from google.cloud import bigquery_datatransfer, pubsub_v1
from google.cloud.bigquery_datatransfer_v1.types import TransferConfig

ACCURACY_QUERY = f"""
    SELECT
      f.sku_id,
      f.date AS forecast_date,
      f.model_display_name,
      f.predicted_sales_qty,
      a.sales_qty AS actual_sales_qty,
      f.predicted_sales_qty - a.sales_qty AS forecast_bias,  -- positive = over-forecast, negative = under-forecast
      ABS(f.predicted_sales_qty - a.sales_qty) AS abs_error,
      SAFE_DIVIDE(ABS(f.predicted_sales_qty - a.sales_qty), a.sales_qty) AS ape,
      CURRENT_TIMESTAMP() AS evaluated_at
    FROM `{PROJECT_ID}.{BQ_DATASET}.forecast_history` f
    JOIN `{PROJECT_ID}.{BQ_DATASET}.actuals` a
      ON f.sku_id = a.sku_id AND f.date = a.date
    WHERE a.date >= DATE_SUB(CURRENT_DATE(), INTERVAL 90 DAY)
"""

if CREATE_ACCURACY_SCHEDULE:
    _publisher = pubsub_v1.PublisherClient()
    _topic_path = _publisher.topic_path(PROJECT_ID, PUBSUB_TOPIC_ID)  # same topic notebook 6 creates

    dts_client = bigquery_datatransfer.DataTransferServiceClient()
    transfer_config = TransferConfig(
        destination_dataset_id=BQ_DATASET,
        display_name="forecast-accuracy-weekly",
        data_source_id="scheduled_query",
        params={
            "query": ACCURACY_QUERY,
            "destination_table_name_template": "forecast_accuracy",
            "write_disposition": "WRITE_TRUNCATE",  # full recompute each run -- simplest correct option
        },
        schedule="every mon 07:00",  # after notebook 6's Monday 06:00 pipeline run has had time to land data
        notification_pubsub_topic=_topic_path,
    )
    created_config = dts_client.create_transfer_config(
        parent=f"projects/{PROJECT_ID}/locations/{REGION}",
        transfer_config=transfer_config,
    )
    print(f"Created scheduled query: {created_config.name}")
else:
    print("CREATE_ACCURACY_SCHEDULE is False -- nothing scheduled. Query above is ready once `actuals` exists:")
    print(ACCURACY_QUERY)

ImportError: cannot import name 'bigquery_datatransfer' from 'google.cloud' (unknown location)